# NB09 — null baselines, calibration skill scores and reliability

**Project:** `BANANA_RIPENESS_MULTISENSOR_Q1`

Uses frozen NB04 and NB06-v1.1 outputs only.

**Addresses:** random baselines, calibration skill scores, mean/median NLL-Brier-ECE, native-vs-diagnostic probabilities, correct reliability diagrams, and a random-selection null.

**Hardware:** CPU only. No model retraining.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings('ignore')

PROJECT = Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
RESULTS_ROOT = PROJECT/'06_RESULTS'
assert PROJECT.exists() and RESULTS_ROOT.exists()

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = RESULTS_ROOT/f'NB09_NULL_CALIBRATION_{RUN_ID}'
TABLE_DIR = RUN_DIR/'tables'
FIG_DIR = RUN_DIR/'figures'
TEXT_DIR = RUN_DIR/'text'
META_DIR = RUN_DIR/'metadata'
for d in [TABLE_DIR,FIG_DIR,TEXT_DIR,META_DIR]:
    d.mkdir(parents=True,exist_ok=True)

ALL_LABELS=[1,2,3,4,5]
K=5
EPS=1e-12
RNG_SEED=20260927
N_NULL=200_000
start_all=time.perf_counter()

print('Project:',PROJECT)
print('Output:',RUN_DIR)
print('Hardware: CPU only')


In [ ]:
LATEST_NB04=RESULTS_ROOT/'LATEST_NB04_RUN.json'
LATEST_NB06=RESULTS_ROOT/'LATEST_NB06_V1_1_RUN.json'
assert LATEST_NB04.exists() and LATEST_NB06.exists()

nb04_ptr=json.loads(LATEST_NB04.read_text(encoding='utf-8'))
nb06_ptr=json.loads(LATEST_NB06.read_text(encoding='utf-8'))
NB04_RUN=Path(nb04_ptr['run_dir'])
NB06_RUN=Path(nb06_ptr['run_dir'])

OUTER_METRICS_PATH=NB04_RUN/'tables'/'outer_loeo_metrics_all_models.csv'
PROB_PATH=NB06_RUN/'predictions'/'outer_probabilities_raw_and_calibrated_v1_1.csv.gz'
CAL_METRICS_PATH=NB06_RUN/'tables'/'outer_calibration_metrics_all_models_v1_1.csv'
SELECTED_CAL_PATH=NB06_RUN/'tables'/'internally_selected_models_calibration_summary_v1_1.csv'

for p in [OUTER_METRICS_PATH,PROB_PATH,CAL_METRICS_PATH,SELECTED_CAL_PATH]:
    assert p.exists(),p
    print('Input:',p)


In [ ]:
def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

input_paths={
    'outer_loeo_metrics_all_models':OUTER_METRICS_PATH,
    'outer_probabilities_raw_and_calibrated_v1_1':PROB_PATH,
    'outer_calibration_metrics_all_models_v1_1':CAL_METRICS_PATH,
    'internally_selected_models_calibration_summary_v1_1':SELECTED_CAL_PATH,
}
pd.DataFrame([
    {'artifact':k,'path':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p)}
    for k,p in input_paths.items()
]).to_csv(META_DIR/'input_hashes.csv',index=False)

outer_metrics=pd.read_csv(OUTER_METRICS_PATH)
prob=pd.read_csv(PROB_PATH)
cal_metrics=pd.read_csv(CAL_METRICS_PATH)
selected_cal=pd.read_csv(SELECTED_CAL_PATH)

assert len(selected_cal)==12
assert outer_metrics.groupby(['outer_test','modality']).size().eq(5).all()
assert len(prob)>0
print('Outer rows:',len(outer_metrics))
print('Probability rows:',f'{len(prob):,}')
print('Selected scenarios:',len(selected_cal))


## Random references

For true-class prevalence \(p_k\) and independent randomized predictions with probability \(q_k\),

\[
E(F1_k)=\frac{2p_kq_k}{p_k+q_k}.
\]

We compute a uniform five-class reference and a development-prevalence reference.


In [ ]:
samples=(prob[['test_id','master_id','stage_true']]
         .drop_duplicates()
         .sort_values(['test_id','master_id'])
         .reset_index(drop=True))

assert samples.groupby('test_id').size().to_dict()=={2:1720,3:4777,4:6133,5:5032}

def reference_metrics(p,q,observed_labels):
    p=np.asarray(p,float); q=np.asarray(q,float)
    f1=np.where((p+q)>0,2*p*q/(p+q),0.0)
    obs_idx=[ALL_LABELS.index(int(c)) for c in observed_labels]
    return {
        'accuracy_expected':float(np.dot(p,q)),
        'macro_f1_expected_observed':float(f1[obs_idx].mean()),
        'macro_f1_expected_fixed_1to5':float(f1.mean()),
        'nll_reference':float(-np.sum(p*np.log(np.clip(q,EPS,1.0)))),
        'brier_reference':float(1-2*np.dot(p,q)+np.dot(q,q)),
        **{f'true_prev_stage_{k}':float(p[k-1]) for k in ALL_LABELS},
        **{f'pred_prob_stage_{k}':float(q[k-1]) for k in ALL_LABELS},
    }

rows=[]
for t in sorted(samples.test_id.unique()):
    test=samples[samples.test_id.eq(t)]
    dev=samples[~samples.test_id.eq(t)]
    p=np.array([(test.stage_true==k).mean() for k in ALL_LABELS])
    q_u=np.repeat(0.2,5)
    q_d=np.array([(dev.stage_true==k).mean() for k in ALL_LABELS])
    observed=sorted(test.stage_true.unique().tolist())
    for name,q in [('uniform',q_u),('development_prevalence',q_d)]:
        rows.append({
            'outer_test':int(t),'reference':name,'n_test':len(test),'n_development':len(dev),
            'observed_labels':','.join(map(str,observed)),
            **reference_metrics(p,q,observed)
        })

references=pd.DataFrame(rows)
references.to_csv(TABLE_DIR/'01_random_reference_by_outer_test.csv',index=False)
display(references[['outer_test','reference','macro_f1_expected_observed',
                    'macro_f1_expected_fixed_1to5','nll_reference','brier_reference']].round(4))


In [ ]:
all_model_mean=(outer_metrics.groupby('outer_test',as_index=False)
    .agg(n_model_modality_rows=('macro_f1_observed','size'),
         mean_macro_f1_observed=('macro_f1_observed','mean'),
         median_macro_f1_observed=('macro_f1_observed','median'),
         mean_balanced_accuracy=('balanced_accuracy','mean'),
         mean_mcc=('mcc','mean')))

u_ref=(references[references.reference.eq('uniform')]
       [['outer_test','macro_f1_expected_observed','macro_f1_expected_fixed_1to5']]
       .rename(columns={'macro_f1_expected_observed':'uniform_macro_f1_observed',
                        'macro_f1_expected_fixed_1to5':'uniform_macro_f1_fixed_1to5'}))

all_model_mean=all_model_mean.merge(u_ref,on='outer_test')
all_model_mean['mean_macro_f1_minus_uniform']=(
    all_model_mean.mean_macro_f1_observed-all_model_mean.uniform_macro_f1_observed)
all_model_mean.to_csv(TABLE_DIR/'02_all_models_mean_vs_uniform_random.csv',index=False)
display(all_model_mean.round(4))


In [ ]:
selected_keys=selected_cal[['outer_test','modality','model']].drop_duplicates()
selected_perf=selected_keys.merge(
    outer_metrics[['outer_test','modality','model','macro_f1_observed',
                   'macro_f1_fixed_1to5','balanced_accuracy','mcc']],
    on=['outer_test','modality','model'],how='left',validate='one_to_one')

u=(references[references.reference.eq('uniform')]
   [['outer_test','macro_f1_expected_observed','macro_f1_expected_fixed_1to5']]
   .rename(columns={'macro_f1_expected_observed':'uniform_macro_f1_observed',
                    'macro_f1_expected_fixed_1to5':'uniform_macro_f1_fixed_1to5'}))
d=(references[references.reference.eq('development_prevalence')]
   [['outer_test','macro_f1_expected_observed','macro_f1_expected_fixed_1to5']]
   .rename(columns={'macro_f1_expected_observed':'devprev_macro_f1_observed',
                    'macro_f1_expected_fixed_1to5':'devprev_macro_f1_fixed_1to5'}))

selected_perf=selected_perf.merge(u,on='outer_test').merge(d,on='outer_test')
selected_perf['selected_minus_uniform_observed']=selected_perf.macro_f1_observed-selected_perf.uniform_macro_f1_observed
selected_perf['selected_minus_devprev_observed']=selected_perf.macro_f1_observed-selected_perf.devprev_macro_f1_observed
selected_perf.to_csv(TABLE_DIR/'03_selected_models_vs_random_references.csv',index=False)
display(selected_perf.round(4))


## Probability validity and skill scores

Diagnostic softmax scores from SVM/PLS are kept transparent but excluded from aggregate probabilistic claims.

\[
Skill=1-Score_{model}/Score_{reference}
\]


In [ ]:
prob_source=(prob.groupby(['test_id','modality','model'],as_index=False)
             .agg(probability_source=('probability_source','first'))
             .rename(columns={'test_id':'outer_test'}))
prob_source['probability_validity']=np.where(
    prob_source.probability_source.astype(str).str.contains('diagnostic',case=False,na=False),
    'diagnostic_only','native_probability')
prob_source.to_csv(TABLE_DIR/'04_probability_source_audit_all_models.csv',index=False)

selected_skill=selected_cal.merge(prob_source,on=['outer_test','modality','model'],
                                  how='left',validate='one_to_one')

for ref_name,short in [('uniform','uniform'),('development_prevalence','devprev')]:
    rr=references[references.reference.eq(ref_name)].set_index('outer_test')
    ref_nll=selected_skill.outer_test.map(rr.nll_reference)
    ref_brier=selected_skill.outer_test.map(rr.brier_reference)
    selected_skill[f'nll_ref_{short}']=ref_nll
    selected_skill[f'brier_ref_{short}']=ref_brier
    selected_skill[f'nll_skill_raw_vs_{short}']=1-selected_skill.nll_raw/ref_nll
    selected_skill[f'nll_skill_cal_vs_{short}']=1-selected_skill.nll_cal/ref_nll
    selected_skill[f'brier_skill_raw_vs_{short}']=1-selected_skill.brier_raw/ref_brier
    selected_skill[f'brier_skill_cal_vs_{short}']=1-selected_skill.brier_cal/ref_brier

selected_skill.to_csv(TABLE_DIR/'05_selected_calibration_skill_scores.csv',index=False)

native=selected_skill[selected_skill.probability_validity.eq('native_probability')].copy()
diagnostic=selected_skill[selected_skill.probability_validity.eq('diagnostic_only')].copy()
assert len(native)==10 and len(diagnostic)==2

summary_rows=[]
for label,df in [('native_selected_only',native),('all_selected_descriptive',selected_skill)]:
    row={'analysis_set':label,'n_scenarios':len(df)}
    for col in ['nll_raw','nll_cal','brier_raw','brier_cal','ece_raw','ece_cal',
                'nll_skill_cal_vs_uniform','nll_skill_cal_vs_devprev',
                'brier_skill_cal_vs_uniform','brier_skill_cal_vs_devprev']:
        row[f'{col}_mean']=float(df[col].mean())
        row[f'{col}_median']=float(df[col].median())
    summary_rows.append(row)

aggregate_summary=pd.DataFrame(summary_rows)
aggregate_summary.to_csv(TABLE_DIR/'06_selected_probability_aggregate_mean_median.csv',index=False)
display(aggregate_summary.T)


In [ ]:
# Random-selection null
scenario_rows=[]
scenario_arrays=[]

for (t,mod),g in outer_metrics.groupby(['outer_test','modality'],sort=True):
    g=g.sort_values('model').reset_index(drop=True)
    scores=g.macro_f1_observed.to_numpy(float)
    inner=g.inner_mean_macro_f1.to_numpy(float)
    si=int(np.argmax(inner)); bi=int(np.argmax(scores))
    scenario_rows.append({
        'outer_test':int(t),'modality':mod,
        'selected_model':g.loc[si,'model'],
        'external_best_model':g.loc[bi,'model'],
        'selected_external_macro_f1':float(scores[si]),
        'external_best_macro_f1':float(scores.max()),
        'external_mean_macro_f1':float(scores.mean()),
        'selection_loss_vs_best':float(scores.max()-scores[si]),
        'selection_loss_vs_mean':float(scores.mean()-scores[si]),
        'winner_match':bool(si==bi)
    })
    scenario_arrays.append(scores)

selection_scenarios=pd.DataFrame(scenario_rows)
selection_scenarios.to_csv(TABLE_DIR/'07_selection_loss_by_scenario.csv',index=False)

score_matrix=np.vstack(scenario_arrays)
best_scores=score_matrix.max(axis=1)
mean_scores=score_matrix.mean(axis=1)
best_indices=score_matrix.argmax(axis=1)

observed_loss_best=float(selection_scenarios.selection_loss_vs_best.mean())
observed_loss_mean=float(selection_scenarios.selection_loss_vs_mean.mean())
observed_matches=int(selection_scenarios.winner_match.sum())

rng=np.random.default_rng(RNG_SEED)
choices=rng.integers(0,5,size=(N_NULL,len(selection_scenarios)))
chosen=np.take_along_axis(score_matrix[None,:,:],choices[:,:,None],axis=2).squeeze(2)

null_loss_best=(best_scores[None,:]-chosen).mean(axis=1)
null_loss_mean=(mean_scores[None,:]-chosen).mean(axis=1)
null_matches=(choices==best_indices[None,:]).sum(axis=1)

p_loss_best_ge=float((np.sum(null_loss_best>=observed_loss_best)+1)/(N_NULL+1))
p_loss_mean_ge=float((np.sum(null_loss_mean>=observed_loss_mean)+1)/(N_NULL+1))
p_matches_le=float((np.sum(null_matches<=observed_matches)+1)/(N_NULL+1))

null_summary=pd.DataFrame([{
    'n_scenarios':12,'algorithms_per_scenario':5,'n_null_draws':N_NULL,'rng_seed':RNG_SEED,
    'observed_mean_loss_vs_external_best':observed_loss_best,
    'null_mean_loss_vs_external_best':float(null_loss_best.mean()),
    'p_null_loss_ge_observed':p_loss_best_ge,
    'observed_mean_loss_vs_model_mean':observed_loss_mean,
    'null_mean_loss_vs_model_mean':float(null_loss_mean.mean()),
    'p_null_loss_vs_mean_ge_observed':p_loss_mean_ge,
    'observed_winner_matches':observed_matches,
    'null_expected_winner_matches':float(null_matches.mean()),
    'p_null_matches_le_observed':p_matches_le
}])
null_summary.to_csv(TABLE_DIR/'08_selection_null_summary.csv',index=False)

pd.DataFrame({
    'mean_loss_vs_external_best':null_loss_best,
    'mean_loss_vs_model_mean':null_loss_mean,
    'winner_matches':null_matches
}).to_csv(TABLE_DIR/'09_selection_null_distribution.csv.gz',index=False,compression='gzip')

display(null_summary.round(6))


In [ ]:
# Correct reliability figure: Spectral modality only, selected native-probability model in each test
PROB_RAW=[f'prob_raw_stage_{c}' for c in ALL_LABELS]
PROB_CAL=[f'prob_cal_v1_1_stage_{c}' for c in ALL_LABELS]

spectral_selected=selected_skill[selected_skill.modality.eq('Spectral')].sort_values('outer_test')
assert len(spectral_selected)==4
assert spectral_selected.probability_validity.eq('native_probability').all()

def reliability_curve(y,p,n_bins=10):
    y=np.asarray(y,int); p=np.asarray(p,float)
    pred=np.array(ALL_LABELS)[np.argmax(p,axis=1)]
    conf=np.max(p,axis=1); corr=(pred==y).astype(float)
    edges=np.linspace(0,1,n_bins+1); out=[]
    for b in range(n_bins):
        lo,hi=edges[b],edges[b+1]
        mask=(conf>=lo)&(conf<(hi if b<n_bins-1 else hi+EPS))
        if mask.any():
            out.append({'bin':b+1,'bin_low':lo,'bin_high':hi,'n':int(mask.sum()),
                        'mean_confidence':float(conf[mask].mean()),
                        'accuracy':float(corr[mask].mean())})
    return pd.DataFrame(out)

rel_rows=[]
fig,axes=plt.subplots(2,2,figsize=(10.5,9),sharex=True,sharey=True)
axes=axes.ravel()

for ax,r in zip(axes,spectral_selected.itertuples()):
    g=prob[(prob.test_id.eq(int(r.outer_test)))&
           (prob.modality.eq(r.modality))&
           (prob.model.eq(r.model))].copy()
    y=g.stage_true.to_numpy(int)
    rr=reliability_curve(y,g[PROB_RAW].to_numpy(float))
    rc=reliability_curve(y,g[PROB_CAL].to_numpy(float))
    for state,z in [('raw',rr),('temperature_scaled_v1_1',rc)]:
        zz=z.copy()
        zz.insert(0,'state',state); zz.insert(0,'model',r.model)
        zz.insert(0,'modality',r.modality); zz.insert(0,'outer_test',int(r.outer_test))
        rel_rows.append(zz)

    ax.plot([0,1],[0,1],'--',linewidth=1,label='Perfect calibration')
    ax.plot(rr.mean_confidence,rr.accuracy,marker='o',label='Raw')
    ax.plot(rc.mean_confidence,rc.accuracy,marker='s',label='Temperature scaled')
    ax.set_title(f'Test {int(r.outer_test)} — {r.model}\nECE raw={r.ece_raw:.3f}; cal={r.ece_cal:.3f}')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.2)

for ax in axes[2:]: ax.set_xlabel('Mean confidence')
for ax in axes[::2]: ax.set_ylabel('Observed accuracy')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=3,bbox_to_anchor=(0.5,-0.005))
fig.suptitle('Reliability under experiment shift — internally selected spectral models',y=1.01)
fig.tight_layout(rect=[0,0.04,1,1])

for ext in ['png','pdf','svg']:
    fig.savefig(FIG_DIR/f'Figure_R1B_reliability_spectral_selected_native.{ext}',
                dpi=300 if ext=='png' else None,bbox_inches='tight')
plt.close(fig)

pd.concat(rel_rows,ignore_index=True).to_csv(
    TABLE_DIR/'10_reliability_bins_spectral_selected_native.csv',index=False)
print('Reliability figure saved.')


In [ ]:
# Figure A: all-model mean Macro-F1 vs uniform random
z=all_model_mean.sort_values('outer_test')
x=np.arange(len(z)); w=0.36
fig,ax=plt.subplots(figsize=(8.2,5.2))
ax.bar(x-w/2,z.mean_macro_f1_observed,width=w,label='Mean of 15 model × modality results')
ax.bar(x+w/2,z.uniform_macro_f1_observed,width=w,label='Uniform-random reference')
ax.set_xticks(x,[f'Test {int(t)}' for t in z.outer_test])
ax.set_ylabel('Macro-F1')
ax.set_title('External discrimination relative to a uniform-random classifier')
ax.legend(); ax.grid(axis='y',alpha=0.2); fig.tight_layout()
for ext in ['png','pdf','svg']:
    fig.savefig(FIG_DIR/f'Figure_R1A_mean_macroF1_vs_uniform_random.{ext}',
                dpi=300 if ext=='png' else None,bbox_inches='tight')
plt.close(fig)

# Figure C: calibrated probabilistic skill, native selected scenarios
pn=native.sort_values(['outer_test','modality']).reset_index(drop=True)
labs=[f'T{int(r.outer_test)} {r.modality}\n{r.model}' for r in pn.itertuples()]
x=np.arange(len(pn)); w=0.38
fig,axes=plt.subplots(2,1,figsize=(11,8),sharex=True)
axes[0].bar(x-w/2,pn.nll_skill_cal_vs_uniform,width=w,label='vs uniform')
axes[0].bar(x+w/2,pn.nll_skill_cal_vs_devprev,width=w,label='vs dev. prevalence')
axes[0].axhline(0,linestyle='--'); axes[0].set_ylabel('NLL skill')
axes[0].set_title('Calibrated probabilistic skill — native probabilities only')
axes[0].legend(); axes[0].grid(axis='y',alpha=0.2)

axes[1].bar(x-w/2,pn.brier_skill_cal_vs_uniform,width=w,label='vs uniform')
axes[1].bar(x+w/2,pn.brier_skill_cal_vs_devprev,width=w,label='vs dev. prevalence')
axes[1].axhline(0,linestyle='--'); axes[1].set_ylabel('Brier skill')
axes[1].set_xticks(x,labs,rotation=50,ha='right')
axes[1].legend(); axes[1].grid(axis='y',alpha=0.2)
fig.tight_layout()
for ext in ['png','pdf','svg']:
    fig.savefig(FIG_DIR/f'Figure_R1C_calibrated_skill_native_selected.{ext}',
                dpi=300 if ext=='png' else None,bbox_inches='tight')
plt.close(fig)

# Figure D: random-selection null
fig,ax=plt.subplots(figsize=(8,5))
ax.hist(null_loss_best,bins=60,density=True,alpha=0.75)
ax.axvline(observed_loss_best,linestyle='--',linewidth=2,label=f'Observed={observed_loss_best:.3f}')
ax.axvline(null_loss_best.mean(),linestyle=':',linewidth=2,label=f'Null mean={null_loss_best.mean():.3f}')
ax.set_xlabel('Mean selection loss vs external post-hoc maximum')
ax.set_ylabel('Density'); ax.set_title('Random-selection null across 12 scenarios')
ax.legend(); fig.tight_layout()
for ext in ['png','pdf','svg']:
    fig.savefig(FIG_DIR/f'Figure_R1D_selection_loss_random_null.{ext}',
                dpi=300 if ext=='png' else None,bbox_inches='tight')
plt.close(fig)


In [ ]:
native_summary=aggregate_summary[aggregate_summary.analysis_set.eq('native_selected_only')].iloc[0]

key={
    'uniform_reference_macro_f1_observed':{
        str(int(r.outer_test)):float(r.macro_f1_expected_observed)
        for r in references[references.reference.eq('uniform')].itertuples()},
    'all_models_mean_macro_f1_observed':{
        str(int(r.outer_test)):float(r.mean_macro_f1_observed)
        for r in all_model_mean.itertuples()},
    'selected_native_probability_scenarios':int(len(native)),
    'selected_diagnostic_softmax_scenarios':int(len(diagnostic)),
    'native_selected_nll_cal_mean':float(native_summary.nll_cal_mean),
    'native_selected_nll_cal_median':float(native_summary.nll_cal_median),
    'native_selected_brier_cal_mean':float(native_summary.brier_cal_mean),
    'native_selected_brier_cal_median':float(native_summary.brier_cal_median),
    'native_selected_ece_cal_mean':float(native_summary.ece_cal_mean),
    'native_selected_ece_cal_median':float(native_summary.ece_cal_median),
    'selection_observed_mean_loss_vs_best':observed_loss_best,
    'selection_null_mean_loss_vs_best':float(null_loss_best.mean()),
    'selection_null_p_loss_ge_observed':p_loss_best_ge,
    'selection_observed_mean_loss_vs_model_mean':observed_loss_mean,
    'selection_observed_winner_matches':observed_matches,
    'selection_null_expected_winner_matches':float(null_matches.mean()),
    'selection_null_p_matches_le_observed':p_matches_le,
    'nll_epsilon_nb06_v1_1':EPS
}
(TABLE_DIR/'11_key_results.json').write_text(json.dumps(key,indent=2),encoding='utf-8')

notes_lines=[
    '# NB09 analysis notes','',
    '## Random references',
    f'Uniform expected observed-class Macro-F1: T2={key["uniform_reference_macro_f1_observed"]["2"]:.3f}, '
    f'T3={key["uniform_reference_macro_f1_observed"]["3"]:.3f}, '
    f'T4={key["uniform_reference_macro_f1_observed"]["4"]:.3f}, '
    f'T5={key["uniform_reference_macro_f1_observed"]["5"]:.3f}.',
    f'All-model mean external Macro-F1: T2={key["all_models_mean_macro_f1_observed"]["2"]:.3f}, '
    f'T3={key["all_models_mean_macro_f1_observed"]["3"]:.3f}, '
    f'T4={key["all_models_mean_macro_f1_observed"]["4"]:.3f}, '
    f'T5={key["all_models_mean_macro_f1_observed"]["5"]:.3f}.','',
    '## Calibration',
    f'Native selected scenarios: {len(native)}; diagnostic-softmax selected scenarios: {len(diagnostic)}.',
    f'After temperature scaling, native-only mean NLL={native_summary.nll_cal_mean:.3f}, '
    f'median NLL={native_summary.nll_cal_median:.3f}.',
    f'Native-only mean Brier={native_summary.brier_cal_mean:.3f}, '
    f'median Brier={native_summary.brier_cal_median:.3f}.',
    f'Native-only mean ECE={native_summary.ece_cal_mean:.3f}, '
    f'median ECE={native_summary.ece_cal_median:.3f}.',
    f'Uniform reference: NLL={math.log(5):.3f}; Brier=0.800.',
    f'NB06-v1.1 log-probability clipping epsilon: {EPS:.0e}.','',
    '## Model-selection null',
    f'Observed mean loss vs external maximum={observed_loss_best:.4f}.',
    f'Random-selection null mean={null_loss_best.mean():.4f}; P(null loss >= observed)={p_loss_best_ge:.4f}.',
    f'Observed winner matches={observed_matches}/12; random expectation={null_matches.mean():.3f}/12; '
    f'P(null matches <= observed)={p_matches_le:.4f}.',
    f'Observed loss relative to model mean={observed_loss_mean:.4f}.','',
    '## Guardrail',
    'The external maximum is a retrospective test-set oracle and is subject to winner-curse optimism. '
    'Selection loss vs the maximum must remain descriptive.'
]
(TEXT_DIR/'analysis_notes.md').write_text('\n'.join(notes_lines)+'\n',encoding='utf-8')

en_lines=[
    '### Candidate results paragraph',
    '',
    'To contextualize performance under experiment shift, external results were compared with uniform '
    'five-class and development-prevalence reference forecasts.',
    f'The uniform-random expected observed-class Macro-F1 values were '
    f'{key["uniform_reference_macro_f1_observed"]["2"]:.3f}, '
    f'{key["uniform_reference_macro_f1_observed"]["3"]:.3f}, '
    f'{key["uniform_reference_macro_f1_observed"]["4"]:.3f}, and '
    f'{key["uniform_reference_macro_f1_observed"]["5"]:.3f} for Tests 2–5.',
    f'The corresponding means across the 15 model-by-modality results were '
    f'{key["all_models_mean_macro_f1_observed"]["2"]:.3f}, '
    f'{key["all_models_mean_macro_f1_observed"]["3"]:.3f}, '
    f'{key["all_models_mean_macro_f1_observed"]["4"]:.3f}, and '
    f'{key["all_models_mean_macro_f1_observed"]["5"]:.3f}.',
    'Aggregate NLL, Brier and ECE summaries were restricted to native probability outputs; diagnostic '
    'softmax transformations of SVM decision scores were reported separately.',
    f'After development-only temperature scaling, native-probability scenarios had mean NLL '
    f'{native_summary.nll_cal_mean:.3f} (median {native_summary.nll_cal_median:.3f}), compared with '
    f'ln(5)={math.log(5):.3f} for a uniform five-class forecast.',
    f'The observed mean selection loss versus the retrospective external maximum was '
    f'{observed_loss_best:.3f}, compared with {null_loss_best.mean():.3f} under random algorithm '
    f'selection; the observed 1/12 winner-match count was not unusual under this null '
    f'(P={p_matches_le:.3f}).'
]
(TEXT_DIR/'candidate_results_paragraph_en.md').write_text('\n'.join(en_lines)+'\n',encoding='utf-8')
print('\n'.join(notes_lines))


In [ ]:
versions={
    'python':platform.python_version(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'matplotlib':plt.matplotlib.__version__,
    'scikit_learn':sklearn.__version__
}
elapsed=time.perf_counter()-start_all

manifest={
    'project':'BANANA_RIPENESS_MULTISENSOR_Q1',
    'notebook':'09_NULL_BASELINES_CALIBRATION_SKILL',
    'run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_nb04_run':str(NB04_RUN),
    'source_nb06_v1_1_run':str(NB06_RUN),
    'classifier_refits':0,'calibration_refits':0,'test_set_retuning':False,
    'null_draws':N_NULL,'rng_seed':RNG_SEED,'nll_clipping_epsilon':EPS,
    'elapsed_minutes':elapsed/60,'versions':versions,'key_results':key
}
(META_DIR/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
(META_DIR/'environment.txt').write_text(
    '\n'.join([f'platform={platform.platform()}',f'cpu_count={os.cpu_count()}',
               *[f'{k}={v}' for k,v in versions.items()]])+'\n',encoding='utf-8')

readme_lines=[
    '# NB09 package','',f'Run: `{RUN_ID}`','',
    '- random and development-prevalence references',
    '- NLL/Brier skill scores',
    '- native-vs-diagnostic probability audit',
    '- mean and median calibration summaries',
    '- random-selection null analysis',
    '- correct banana reliability figure',
    '- manuscript-ready notes','',
    'No classifier or calibrator was refitted.'
]
(RUN_DIR/'README_RESULTS.md').write_text('\n'.join(readme_lines)+'\n',encoding='utf-8')

hash_rows=[]
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file() and p.name!='package_hashes.csv':
        hash_rows.append({'relative_path':str(p.relative_to(RUN_DIR)),
                          'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(hash_rows).to_csv(META_DIR/'package_hashes.csv',index=False)

zip_base=RESULTS_ROOT/f'RESULTS_NB09_NULL_CALIBRATION_{RUN_ID}'
zip_path=Path(shutil.make_archive(str(zip_base),'zip',
                                  root_dir=RUN_DIR.parent,base_dir=RUN_DIR.name))

latest={'notebook':'09_NULL_BASELINES_CALIBRATION_SKILL',
        'run_id':RUN_ID,'run_dir':str(RUN_DIR),'zip_path':str(zip_path),
        'zip_bytes':zip_path.stat().st_size,'elapsed_minutes':elapsed/60}
(RESULTS_ROOT/'LATEST_NB09_RUN.json').write_text(json.dumps(latest,indent=2),encoding='utf-8')

print('\n✅ NB09 COMPLETED')
print('Run directory:',RUN_DIR)
print('ZIP:',zip_path)
print('Elapsed (min):',round(elapsed/60,2))
print('Send the ZIP back to ChatGPT.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Automatic download not completed:',repr(e))
    print('ZIP is safely stored in Drive:',zip_path)
